# Limpieza, análisis y estructuración de datos con Pandas (Instacart)

**Actividad GA1-220501094-01-AA1-EV06**

**Pregunta orientadora:** ¿Cómo preparar y estructurar los datos de Instacart para obtener información confiable sobre los hábitos de compra?

Contenido:

- Lección 1: carga de datos (preparación)
- Lección 2: limpieza y estandarización
- Lección 3: filtrado, ordenamiento y agrupación
- Lección 4: combinación de datasets

## Preparación del entorno

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RUTA = Path('../data/raw')
SALIDA = Path('../data/processed')
SALIDA.mkdir(exist_ok=True)
print(pd.__version__)

3.0.6


## Lección 1: carga de los cinco archivos

Los CSV usan `;` como separador, por eso se indica `sep=';'`.

In [2]:
orders      = pd.read_csv(RUTA / 'instacart_orders.csv', sep=';')
order_prods = pd.read_csv(RUTA / 'order_products.csv', sep=';')
products    = pd.read_csv(RUTA / 'products.csv', sep=';')
aisles      = pd.read_csv(RUTA / 'aisles.csv', sep=';')
departments = pd.read_csv(RUTA / 'departments.csv', sep=';')

In [3]:
for nombre, df in [('orders', orders), ('order_prods', order_prods), ('products', products),
                   ('aisles', aisles), ('departments', departments)]:
    print(nombre, df.shape)

orders (478967, 6)
order_prods (4545007, 4)
products (49694, 4)
aisles (134, 2)
departments (21, 2)


Resultado esperado:

```
orders (478967, 6)
order_prods (4545007, 4)
products (49694, 4)
aisles (134, 2)
departments (21, 2)
```

## Lección 2: limpieza y estandarización

Orden de trabajo: duplicados, tipos, nulos, texto, duplicados implícitos y verificación. Para cada problema la decisión es **eliminar, corregir o conservar**, según lo que signifique el dato.

### 2.1 Duplicados exactos

In [4]:
duplicados = orders[orders.duplicated(keep=False)].sort_values('order_id')
print(len(duplicados))
duplicados[['order_id', 'order_dow', 'order_hour_of_day']].head(6)

30


,order_id,order_dow,order_hour_of_day
354993,391768,3,2
371905,391768,3,2
119251,408114,3,2
321100,408114,3,2
394347,467134,3,2
250626,467134,3,2


**Interpretación:** hay 30 filas (15 pedidos y sus 15 copias). Todas ocurren el miércoles a las 2 a. m., lo que apunta a un fallo del sistema y no a pedidos reales. **Decisión: eliminar.**

In [5]:
orders = orders.drop_duplicates().reset_index(drop=True)
print(orders.shape, orders['order_id'].is_unique)

(478952, 6) True


### 2.2 Tipos de dato con `astype`

In [6]:
print('Antes:', round(orders.memory_usage(deep=True).sum() / 1e6, 1), 'MB')

orders = orders.astype({
    'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
    'order_dow': 'int8', 'order_hour_of_day': 'int8',
    'days_since_prior_order': 'Int16',
})
order_prods = order_prods.astype({
    'order_id': 'int32', 'product_id': 'int32',
    'add_to_cart_order': 'Int16', 'reordered': 'int8',
})

print('Después:', round(orders.memory_usage(deep=True).sum() / 1e6, 1), 'MB')

Antes: 23.0 MB


Después: 7.2 MB


**Interpretación:** al usar el tipo entero más pequeño que alcanza para cada columna, `orders` baja de 23,0 MB a 7,2 MB. Las columnas con nulos usan `Int16` (con mayúscula), que admite enteros y nulos a la vez.

### 2.3 Nulos: comprobar la causa antes de decidir

In [7]:
nulos = orders['days_since_prior_order'].isna()
primeros = orders['order_number'] == 1
print((nulos == primeros).all())

True


**Interpretación:** cada nulo de `days_since_prior_order` es un primer pedido: no existe un pedido anterior. **Decisión: conservar el nulo.** Rellenar con 0 inventaría que el cliente volvió el mismo día.

In [8]:
print(products[products['product_name'].isna()][['aisle_id', 'department_id']].value_counts())
products['product_name'] = products['product_name'].fillna('Unknown')
print(products['product_name'].isna().sum())

aisle_id  department_id
100       21               1258
Name: count, dtype: int64
0


**Interpretación:** los 1 258 productos sin nombre pertenecen todos al pasillo 100 y al departamento 21 (`missing`). El producto existe, así que no se borra. **Decisión: corregir** con el valor explícito `'Unknown'`.

| Columna | Causa | Decisión |
| --- | --- | --- |
| `days_since_prior_order` | Primer pedido del cliente | Conservar el nulo |
| `product_name` | Producto sin datos de catálogo | Rellenar con `'Unknown'` |
| `add_to_cart_order` | Se investiga en el Reto 2 | Conservar el nulo |

### 2.4 Texto con los métodos `.str`

In [9]:
nombres = products['product_name']
print('Espacios dobles:', nombres.str.contains('  ', regex=False).sum())
print('Espacio invisible:', nombres.str.contains('\xa0', regex=False).sum())

products['product_name'] = (
    products['product_name']
    .str.replace('\xa0', ' ', regex=False)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
    .str.title()
)
print(products['product_name'].str.contains('  ', regex=False).sum(),
      products['product_name'].str.contains('\xa0', regex=False).sum())

Espacios dobles: 403
Espacio invisible: 16
0 0


**Interpretación:** había 403 nombres con espacios dobles y 16 con espacio invisible (`\xa0`). Tras la limpieza quedan 0 de cada uno, y los nombres quedan en formato título.

### 2.5 Duplicados implícitos

In [10]:
conocidos = products[products['product_name'] != 'Unknown']
repetidos = conocidos[conocidos['product_name'].duplicated(keep=False)]
print(conocidos['product_name'].duplicated().sum())
print(repetidos['product_id'].isin(order_prods['product_id']).sum(), 'de', len(repetidos))

141
268 de 280


**Interpretación:** quedan 141 nombres repetidos con `product_id` distintos, y casi todos aparecen en pedidos. **Decisión: conservar y documentar.** Borrarlos dejaría pedidos apuntando a productos inexistentes; en los análisis se agrupa por `product_id`, no por nombre.

### 2.6 Limpieza reutilizable con funciones

In [11]:
def limpiar_orders(df):
    df = df.drop_duplicates().reset_index(drop=True)
    df = df.astype({'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
                    'order_dow': 'int8', 'order_hour_of_day': 'int8',
                    'days_since_prior_order': 'Int16'})
    assert df['order_id'].is_unique
    return df

def limpiar_products(df):
    df = df.copy()
    df['product_name'] = (df['product_name'].fillna('Unknown')
                          .str.replace('\xa0', ' ', regex=False)
                          .str.replace(r'\s+', ' ', regex=True)
                          .str.strip().str.title())
    return df

print(limpiar_orders(orders).shape)

(478952, 6)


In [12]:
assert orders.shape == (478952, 6)
assert orders['days_since_prior_order'].isna().sum() == 28817   # 2 nulos estaban entre los duplicados
assert products['product_name'].isna().sum() == 0
assert str(order_prods['add_to_cart_order'].dtype) == 'Int16'
print('Verificación superada')

Verificación superada


### Laboratorio guiado 2: pipeline de limpieza con informe de cambios

**Encargo:** construir una secuencia de pasos de limpieza encadenados que parta del archivo crudo, se ejecute con una sola instrucción y deje un informe de lo que cambió en cada paso.

**Paso 1: recargar los datos crudos**

In [13]:
orders_crudo   = pd.read_csv(RUTA / 'instacart_orders.csv', sep=';')
products_crudo = pd.read_csv(RUTA / 'products.csv', sep=';')
op_crudo       = pd.read_csv(RUTA / 'order_products.csv', sep=';')
print(orders_crudo.shape, products_crudo.shape, op_crudo.shape)

(478967, 6) (49694, 4) (4545007, 4)


**Paso 2: función que anota el estado de la tabla**

In [14]:
registro = []

def anotar(df, tabla, paso):
    registro.append({
        'tabla': tabla,
        'paso': paso,
        'filas': len(df),
        'nulos': int(df.isna().sum().sum()),
        'memoria_mb': round(df.memory_usage(deep=True).sum() / 1e6, 1),
    })
    return df

**Paso 3: un paso de limpieza por función**

In [15]:
def quitar_duplicados(df):
    return df.drop_duplicates().reset_index(drop=True)

def tipos_orders(df):
    return df.astype({'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
                      'order_dow': 'int8', 'order_hour_of_day': 'int8',
                      'days_since_prior_order': 'Int16'})

def limpiar_nombres(df):
    df = df.copy()
    df['product_name'] = (df['product_name'].fillna('Unknown')
                          .str.replace('\xa0', ' ', regex=False)
                          .str.replace(r'\s+', ' ', regex=True)
                          .str.strip().str.title())
    return df

def tipos_order_prods(df):
    return df.astype({'order_id': 'int32', 'product_id': 'int32',
                      'add_to_cart_order': 'Int16', 'reordered': 'int8'})

**Paso 4: encadenar los pasos con `pipe`**

In [16]:
orders_limpio = (orders_crudo
    .pipe(anotar, 'orders', '0 crudo')
    .pipe(quitar_duplicados)
    .pipe(anotar, 'orders', '1 sin duplicados')
    .pipe(tipos_orders)
    .pipe(anotar, 'orders', '2 tipos'))

products_limpio = (products_crudo
    .pipe(anotar, 'products', '0 crudo')
    .pipe(limpiar_nombres)
    .pipe(anotar, 'products', '1 nombres'))

op_limpio = (op_crudo
    .pipe(anotar, 'order_prods', '0 crudo')
    .pipe(tipos_order_prods)
    .pipe(anotar, 'order_prods', '1 tipos'))

**Paso 5: informe de cambios**

In [17]:
informe = pd.DataFrame(registro)
informe

,tabla,paso,filas,nulos,memoria_mb
0,orders,0 crudo,478967,28819,23.0
1,orders,1 sin duplicados,478952,28817,23.0
2,orders,2 tipos,478952,28817,7.2
3,products,0 crudo,49694,1258,3.1
4,products,1 nombres,49694,0,3.1
5,order_prods,0 crudo,4545007,836,145.4
6,order_prods,1 tipos,4545007,836,54.5


**Interpretación del informe:**

- Quitar duplicados eliminó 15 filas y 2 nulos: dos de los pedidos repetidos eran primeros pedidos.
- Corregir tipos no cambió filas ni nulos, pero redujo la memoria de `orders` de 23,0 a 7,2 MB y la de `order_prods` de 145,4 a 54,5 MB.
- Los 836 nulos de `order_prods` siguen ahí a propósito: se decidió conservarlos (ver Reto 2, punto 3).

**Paso 6: validar el resultado**

In [18]:
def validar(o, p, op):
    assert o['order_id'].is_unique, 'order_id repetido'
    assert o['order_hour_of_day'].between(0, 23).all(), 'hora fuera de rango'
    assert p['product_name'].notna().all(), 'productos sin nombre'
    assert not p['product_name'].str.contains('\xa0', regex=False).any(), 'espacios invisibles'
    assert op['reordered'].isin([0, 1]).all(), 'reordered con valores raros'
    return 'Validación superada'

print(validar(orders_limpio, products_limpio, op_limpio))
print(orders_limpio.equals(orders), products_limpio.equals(products))

Validación superada
True True


**Conclusiones del laboratorio:** el pipeline produce exactamente las mismas tablas que la limpieza manual (`True True`), deja un informe con cifras de cada paso y termina con una validación automática de las reglas de calidad.

### Reto 2

**Ejercicio 1.** Duplicados exactos en `order_prods`, pares `order_id` + `product_id` repetidos y valores de `reordered`.

In [19]:
print('Filas duplicadas exactas:', order_prods.duplicated().sum())
print('Pares order_id + product_id repetidos:', order_prods.duplicated(['order_id', 'product_id']).sum())
print('Valores de reordered:', sorted(order_prods['reordered'].unique().tolist()))

Filas duplicadas exactas: 0


Pares order_id + product_id repetidos: 0
Valores de reordered: [0, 1]


**Respuesta:** no hay filas duplicadas, no se repite ningún par `order_id` + `product_id` y `reordered` solo contiene 0 y 1. `order_prods` no necesita limpieza de duplicados.

**Ejercicio 2.** Nombres duplicados o con espacios en `aisles` y `departments`.

In [20]:
for nombre, df, col in [('aisles', aisles, 'aisle'), ('departments', departments, 'department')]:
    s = df[col]
    print(nombre,
          '| duplicados:', s.duplicated().sum(),
          '| espacios al inicio o final:', (s != s.str.strip()).sum(),
          '| espacios dobles:', s.str.contains('  ', regex=False).sum(),
          '| con mayúsculas:', (s != s.str.lower()).sum())

aisles | duplicados: 0 | espacios al inicio o final: 0 | espacios dobles: 0 | con mayúsculas: 0
departments | duplicados: 0 | espacios al inicio o final: 0 | espacios dobles: 0 | con mayúsculas: 0


**Respuesta:** ninguna de las dos tablas tiene nombres duplicados, espacios sobrantes ni mayúsculas mezcladas. **No necesitan limpieza de texto.**

**Ejercicio 3.** Los 836 nulos de `add_to_cart_order`.

In [21]:
nulos_pos = order_prods[order_prods['add_to_cart_order'].isna()]
tam_pedido = order_prods.groupby('order_id').size()
ids = nulos_pos['order_id'].unique()

print('Nulos:', len(nulos_pos))
print('Pedidos con nulos:', len(ids))
print('Productos por pedido (mínimo y máximo):', tam_pedido[ids].min(), tam_pedido[ids].max())
print('Valor máximo de add_to_cart_order:', order_prods['add_to_cart_order'].max())
print('Pedidos con más de 64 productos:', (tam_pedido > 64).sum())

Nulos: 836
Pedidos con nulos: 70
Productos por pedido (mínimo y máximo): 65 127


Valor máximo de add_to_cart_order: 64
Pedidos con más de 64 productos: 70


**Respuesta:** los 836 nulos están en 70 pedidos, y cada uno de esos pedidos tiene como mínimo 65 productos (el máximo es 127). El valor más alto que aparece en `add_to_cart_order` es 64, y los 70 pedidos con más de 64 productos son exactamente los que tienen nulos. **Explicación:** el sistema solo numera hasta la posición 64; los productos que pasan de ese límite quedan sin posición registrada. **Decisión: conservar el nulo.** La posición real es desconocida y rellenarla inventaría un dato.

**Ejercicio 4.** Nombres que cambiaron con la estandarización.

In [22]:
products_crudo = pd.read_csv(RUTA / 'products.csv', sep=';')
con_nombre = products_crudo['product_name'].notna()
cambiaron = (products_crudo.loc[con_nombre, 'product_name'] != products.loc[con_nombre, 'product_name']).sum()
print('Nombres no nulos que cambiaron:', cambiaron)

Nombres no nulos que cambiaron: 6321


**Respuesta:** cambiaron 6 321 nombres: 5 902 solo por el cambio a formato título y 419 por espacios dobles o invisibles (403 + 16).

**Ejercicio 5.** Nombre del día como categoría ordenada y comparación de memoria.

In [23]:
dias = {0: 'domingo', 1: 'lunes', 2: 'martes', 3: 'miércoles',
        4: 'jueves', 5: 'viernes', 6: 'sábado'}
dia_texto = orders['order_dow'].map(dias)
dia_cat = pd.Series(pd.Categorical(dia_texto, categories=list(dias.values()), ordered=True))

print('Texto:    ', round(dia_texto.memory_usage(deep=True) / 1e6, 2), 'MB')
print('Categoría:', round(dia_cat.memory_usage(deep=True) / 1e6, 2), 'MB')
print(dia_cat.min(), '<', dia_cat.max())

Texto:     7.14 MB
Categoría: 0.48 MB
domingo < sábado


**Respuesta:** como texto ocupa 7,14 MB y como categoría 0,48 MB, unas 15 veces menos, porque la categoría guarda cada nombre una sola vez y usa códigos pequeños. Además, al ser ordenada permite comparar y ordenar los días de domingo a sábado.

**Ejercicio 6.** Filas de `order_prods` de productos `'Unknown'`.

In [24]:
ids_unknown = products.loc[products['product_name'] == 'Unknown', 'product_id']
filas_unknown = order_prods[order_prods['product_id'].isin(ids_unknown)]
print('Filas:', len(filas_unknown))
print('Pedidos distintos:', filas_unknown['order_id'].nunique())
print('Porcentaje de las filas:', round(len(filas_unknown) / len(order_prods) * 100, 2), '%')

Filas: 9547
Pedidos distintos: 8254
Porcentaje de las filas: 0.21 %


**Respuesta:** hay 9 547 filas (0,21 % del total) en 8 254 pedidos. **No los eliminaría:** son compras reales, así que borrarlos quitaría productos de pedidos legítimos y alteraría el tamaño de esos carritos y las tasas de recompra. Se conservan marcados como `'Unknown'` y se pueden excluir solo en los análisis donde importe el nombre.

**Ejercicio 7.** `'missing'` como nulo real en una copia de `departments`.

In [25]:
departments_copia = departments.copy()
departments_copia['department'] = departments_copia['department'].replace('missing', pd.NA)
print(departments_copia.tail(2))
print('Nulos en la copia:', departments_copia['department'].isna().sum())
print('Nulos en la original:', departments['department'].isna().sum())

    department_id department
19             20       deli
20             21        NaN


Nulos en la copia: 1
Nulos en la original: 0


**Respuesta:** en la copia, el departamento 21 queda como nulo real y la original no cambia. No conviene hacerlo en la original antes de agrupar porque `groupby` descarta por defecto las filas con clave nula: los productos del departamento `missing` desaparecerían de los conteos sin avisar.

### Documentación de resultados de limpieza (Lección 2)

| Tabla | Problema | Evidencia | Decisión |
| --- | --- | --- | --- |
| `orders` | Filas duplicadas | 15 pedidos repetidos (miércoles, 2 a. m.) | Eliminar |
| `orders` | Tipos sobredimensionados | 23,0 MB | Reducir a 7,2 MB con `astype` |
| `orders` | Nulos en `days_since_prior_order` | 28 817 (todos son primer pedido) | Conservar |
| `products` | Nulos en `product_name` | 1 258 | Rellenar con `'Unknown'` |
| `products` | Espacios dobles e invisibles | 403 y 16 | Estandarizar texto |
| `products` | Nombres repetidos con `product_id` distinto | 141 | Conservar y agrupar por `product_id` |
| `order_prods` | Nulos en `add_to_cart_order` | 836 en 70 pedidos de más de 64 productos | Conservar |
| `order_prods` | Tipos sobredimensionados | 145,4 MB | Reducir a 54,5 MB |
| `aisles`, `departments` | Sin problemas de texto ni duplicados | 0 | Sin cambios |

## Lección 3: filtrado, ordenamiento y agrupación

Antes de escribir código, cada pregunta se traduce a la frase "*métrica* por *grupo*". Por ejemplo, "¿a qué hora se compra más?" se convierte en "número de pedidos por hora".

### 3.1 Filtrado con máscaras booleanas

In [26]:
es_primer_pedido = orders['order_number'] == 1
print(es_primer_pedido.head(3))
primeros = orders[es_primer_pedido]
print(len(primeros))

0    False
1    False
2    False
Name: order_number, dtype: bool
28817


In [27]:
es_fin_de_semana = orders['order_dow'].isin([0, 6])
es_manana = orders['order_hour_of_day'].between(6, 11)
print(len(orders[es_fin_de_semana & es_manana]))

45599


**Interpretación:** hay 28 817 primeros pedidos, el mismo número de nulos de `days_since_prior_order`. Combinar máscaras con `&` permite contar los pedidos de fin de semana por la mañana.

### 3.2 Filtrado con `query`

In [28]:
nocturnos_30 = orders.query('order_hour_of_day >= 22 and days_since_prior_order == 30')
print(len(nocturnos_30))

1449


**Interpretación:** hay 1 449 pedidos hechos desde las 10 p. m. por clientes que tardaron 30 días en volver.

### 3.3 Ordenamiento con `sort_values` y `nlargest`

In [29]:
print(orders.nlargest(3, 'order_number')[['user_id', 'order_number']])
historial = orders.sort_values(['user_id', 'order_number'])
print(historial[['user_id', 'order_number']].head(3))

      user_id  order_number
2482     8220           100
4552   164906           100
6402   172294           100


        user_id  order_number
237012        2             1
317136        2             4
337239        4             2


**Interpretación:** tres clientes llegaron al pedido número 100. En el historial ordenado, el cliente 2 aparece primero, con sus pedidos 1 y 4.

### 3.4 Agrupación con `groupby`

In [30]:
pedidos_por_hora = orders.groupby('order_hour_of_day')['order_id'].count()
print(pedidos_por_hora.idxmax(), pedidos_por_hora.max())

10 40578


**Interpretación:** la hora pico es las 10 a. m., con 40 578 pedidos.

In [31]:
resumen_dia = (orders
    .groupby('order_dow')
    .agg(pedidos=('order_id', 'count'),
         clientes=('user_id', 'nunique'),
         espera_media=('days_since_prior_order', 'mean'))
    .round(1))
resumen_dia

,pedidos,clientes,espera_media
order_dow,,,
0,84090,57661,11.8
1,82185,56479,11.3
2,65833,48587,11.2
3,60897,45331,10.7
4,59810,44281,10.5
5,63488,46127,10.5
6,62649,45604,11.4


**Interpretación:** domingo (84 090) y lunes (82 185) concentran más pedidos; jueves es el día con menos (59 810). La espera media entre pedidos se mueve en un rango corto, de 10,5 a 11,8 días.

### 3.5 Tabla dinámica con `pivot_table`

In [32]:
matriz = orders.pivot_table(index='order_hour_of_day', columns='order_dow',
                            values='order_id', aggfunc='count')
print(matriz.shape)
matriz.loc[[8, 10, 14, 20]]

(24, 7)


order_dow,0,1,2,3,4,5,6
order_hour_of_day,,,,,,,
8,3908,4751,3485,3125,3136,3373,3246
10,6978,7881,5489,5026,4938,5347,4919
14,7533,6545,5344,4774,4795,5265,5375
20,2544,2332,2061,1917,2037,1886,1847


**Interpretación:** la tabla tiene 24 horas por 7 días. La celda más alta de las mostradas es el lunes a las 10 h, con 7 881 pedidos.

### 3.6 Valores por fila con `transform`

In [33]:
orders['pedidos_cliente'] = orders.groupby('user_id')['order_id'].transform('count')
frecuentes = orders[orders['pedidos_cliente'] >= 5]
print(len(frecuentes), frecuentes['user_id'].nunique())
orders = orders.drop(columns='pedidos_cliente')
print(orders.shape)

231946 30527
(478952, 6)


**Interpretación:** 30 527 clientes tienen 5 pedidos o más, y entre todos suman 231 946 pedidos. Se elimina la columna auxiliar para dejar `orders` con sus 6 columnas.

### Laboratorio guiado 3: perfil de hábitos de compra

**Encargo:** mercadeo quiere campañas distintas según qué tan seguido compra cada cliente y necesita saber **a qué hora compra cada tipo de cliente**.

**Paso 1: pedidos por cliente**

In [34]:
pedidos_por_cliente = orders.groupby('user_id')['order_id'].count()
print(len(pedidos_por_cliente))
pedidos_por_cliente.describe()

157437


count    157437.000000
mean          3.042182
std           2.746842
min           1.000000
25%           1.000000
50%           2.000000
75%           4.000000
max          28.000000
Name: order_id, dtype: float64

**Interpretación:** hay 157 437 clientes; la mitad tiene 2 pedidos o menos y el máximo es 28. Los segmentos se definen con los pedidos que aparecen en este conjunto de datos.

**Paso 2: segmentar clientes con `pd.cut`** (ocasional: 1 pedido, regular: 2 a 4, frecuente: 5 o más)

In [35]:
segmento_cliente = pd.cut(pedidos_por_cliente,
                          bins=[0, 1, 4, np.inf],
                          labels=['ocasional', 'regular', 'frecuente'])
print(segmento_cliente.value_counts().sort_index())
print((segmento_cliente.value_counts(normalize=True).sort_index() * 100).round(1))

order_id
ocasional    55357
regular      71553
frecuente    30527
Name: count, dtype: int64
order_id
ocasional    35.2
regular      45.4
frecuente    19.4
Name: proportion, dtype: float64


**Interpretación:** el 35,2 % de los clientes son ocasionales (55 357), el 45,4 % regulares (71 553) y el 19,4 % frecuentes (30 527).

**Paso 3: llevar el segmento a cada pedido con `map`**

In [36]:
orders['segmento'] = orders['user_id'].map(segmento_cliente)
orders[['user_id', 'order_number', 'segmento']].head()

,user_id,order_number,segmento
0,183418,11,regular
1,163593,5,regular
2,39980,4,ocasional
3,82516,56,frecuente
4,196724,2,ocasional


**Paso 4: resumen por segmento**

In [37]:
resumen_segmento = (orders
    .groupby('segmento', observed=True)
    .agg(clientes=('user_id', 'nunique'),
         pedidos=('order_id', 'count'),
         espera_mediana=('days_since_prior_order', 'median')))
resumen_segmento

,clientes,pedidos,espera_mediana
segmento,,,
ocasional,55357,55357,16.0
regular,71553,191649,11.0
frecuente,30527,231946,6.0


**Interpretación:** el 19,4 % de los clientes (los frecuentes) hace 231 946 de los 478 952 pedidos (48,4 %) y vuelve con una mediana de 6 días. Los ocasionales tienen una mediana de 16 días.

**Paso 5: franjas horarias y tabla cruzada**

In [38]:
orders['franja'] = pd.cut(orders['order_hour_of_day'],
                          bins=[-1, 5, 11, 17, 23],
                          labels=['madrugada', 'mañana', 'tarde', 'noche'])

perfil = (pd.crosstab(orders['segmento'], orders['franja'], normalize='index')
            .mul(100).round(1))
perfil

franja,madrugada,mañana,tarde,noche
segmento,,,,
ocasional,2.1,30.0,49.4,18.5
regular,1.9,31.6,48.4,18.1
frecuente,1.7,35.2,45.9,17.2


**Interpretación:** todos los segmentos compran sobre todo en la tarde, pero los frecuentes compran más en la mañana (35,2 %) que los ocasionales (30,0 %). Esto sugiere enviar las campañas de fidelización por la mañana.

**Paso 6: combinaciones de día y franja más activas**

In [39]:
(orders.groupby(['order_dow', 'franja'], observed=True)
       .size()
       .sort_values(ascending=False)
       .head(3))

order_dow  franja
0          tarde     42340
1          tarde     37432
2          tarde     30751
dtype: int64

**Interpretación:** las combinaciones más activas son domingo en la tarde (42 340 pedidos), lunes en la tarde (37 432) y martes en la tarde (30 751).

**Paso 7: dejar `orders` como estaba**

In [40]:
orders = orders.drop(columns=['segmento', 'franja'])
print(orders.shape)

(478952, 6)


### Reto 3

**Ejercicio 1.** *Pregunta:* pedidos por hora, solo los domingos (`order_dow == 0`). Las tres horas con más pedidos.

In [41]:
domingos = orders[orders['order_dow'] == 0]
domingos.groupby('order_hour_of_day')['order_id'].count().nlargest(3)

order_hour_of_day
15    7585
14    7533
13    7483
Name: order_id, dtype: int64

**Respuesta:** los domingos las horas con más pedidos son las 15 h (7 585), las 14 h (7 533) y las 13 h (7 483): el pico es en la tarde.

**Ejercicio 2.** *Pregunta:* espera media (`days_since_prior_order`) por hora. ¿Dónde es mayor y dónde menor?

In [42]:
espera_hora = orders.groupby('order_hour_of_day')['days_since_prior_order'].mean()
print('Mayor:', espera_hora.idxmax(), 'h ->', round(float(espera_hora.max()), 2), 'días')
print('Menor:', espera_hora.idxmin(), 'h ->', round(float(espera_hora.min()), 2), 'días')

Mayor: 0 h -> 12.52 días
Menor: 22 h -> 10.62 días


**Respuesta:** la espera media es mayor a las 0 h (12,52 días) y menor a las 22 h (10,62 días), una diferencia de casi 2 días.

**Ejercicio 3.** *Pregunta:* clientes con 10 pedidos o más (pedidos por cliente, con condición).

In [43]:
pedidos_cliente = orders.groupby('user_id')['order_id'].count()
print('Clientes con 10 pedidos o más:', (pedidos_cliente >= 10).sum())
print('Máximo de pedidos de un cliente:', pedidos_cliente.max())

Clientes con 10 pedidos o más: 6327
Máximo de pedidos de un cliente: 28


**Respuesta:** 6 327 clientes tienen 10 pedidos o más; el máximo es 28.

**Ejercicio 4.** *Condición:* `days_since_prior_order == 7`. Cantidad y porcentaje del total.

In [44]:
a_los_7 = orders['days_since_prior_order'] == 7
print('Pedidos a los 7 días:', a_los_7.sum())
print('% del total de pedidos:', round(a_los_7.sum() / len(orders) * 100, 2))
print('% de los pedidos con valor de espera:',
      round(a_los_7.sum() / orders['days_since_prior_order'].notna().sum() * 100, 2))

Pedidos a los 7 días: 44577
% del total de pedidos: 9.31
% de los pedidos con valor de espera: 9.9


**Respuesta:** 44 577 pedidos se hicieron 7 días después del anterior: 9,31 % del total de pedidos, o 9,9 % si solo se cuentan los que tienen valor de espera (los primeros pedidos no tienen).

**Ejercicio 5.** *Pregunta:* productos por pedido, en `order_prods`. Pedidos con 20 productos o más y pedido más grande.

In [45]:
productos_por_pedido = order_prods.groupby('order_id').size()
print('Pedidos con 20 productos o más:', (productos_por_pedido >= 20).sum())
print('Pedido más grande:', productos_por_pedido.idxmax(), 'con', productos_por_pedido.max(), 'productos')

Pedidos con 20 productos o más: 49420
Pedido más grande: 61355 con 127 productos


**Respuesta:** 49 420 pedidos tienen 20 productos o más. El pedido más grande es el 61355, con 127 productos.

**Ejercicio 6.** *Pregunta:* tasa de recompra (media de `reordered`) por posición del carrito, posiciones 1 a 5.

In [46]:
primeras = order_prods[order_prods['add_to_cart_order'].between(1, 5)]
primeras.groupby('add_to_cart_order')['reordered'].mean().round(3)

add_to_cart_order
1    0.679
2    0.677
3    0.659
4    0.638
5    0.617
Name: reordered, dtype: float64

**Respuesta:** la tasa de recompra baja con la posición, de 0,679 en la posición 1 a 0,617 en la 5. Los primeros productos que los clientes agregan al carrito son los que más repiten.

**Ejercicio 7.** *Pregunta:* mediana de `days_since_prior_order` por tipo de día (fin de semana o resto).

In [47]:
orders['es_fin_de_semana'] = orders['order_dow'].isin([0, 6])
print(orders.groupby('es_fin_de_semana')['days_since_prior_order'].median())
orders = orders.drop(columns='es_fin_de_semana')

es_fin_de_semana
False    7.0
True     8.0
Name: days_since_prior_order, dtype: Float64


**Respuesta:** la mediana de espera es de 8 días en pedidos de fin de semana y de 7 días el resto de la semana.

## Lección 4: combinación de datasets

`merge` une **columnas** de dos tablas emparejando sus claves; `concat` apila **filas** de tablas con las mismas columnas. Un `merge` mal hecho no da error: multiplica o pierde filas en silencio, por eso cada unión se comprueba.

### 4.1 Tipos de unión con el parámetro `how`

In [48]:
pedidos = pd.DataFrame({'order_id': [1, 2, 3], 'user_id': [10, 20, 30]})
detalle = pd.DataFrame({'order_id': [1, 1, 2, 4], 'product_id': [100, 200, 100, 300]})

for tipo in ['inner', 'left', 'outer']:
    resultado = pedidos.merge(detalle, on='order_id', how=tipo)
    print(tipo, len(resultado))
    print(resultado, '\n')

inner 3
   order_id  user_id  product_id
0         1       10         100
1         1       10         200
2         2       20         100 

left 4
   order_id  user_id  product_id
0         1       10       100.0
1         1       10       200.0
2         2       20       100.0
3         3       30         NaN 

outer 5
   order_id  user_id  product_id
0         1     10.0       100.0
1         1     10.0       200.0
2         2     20.0       100.0
3         3     30.0         NaN
4         4      NaN       300.0 



**Interpretación:** `inner` conserva solo las claves que están en ambas tablas (3 filas); `left` conserva todos los pedidos, incluido el 3 sin productos (4 filas); `outer` conserva todo, incluido el pedido 4 que no tiene cliente (5 filas). El pedido 1 aparece dos veces porque tiene dos productos, y eso es correcto.

### 4.2 Validación de la relación con `validate`

In [49]:
catalogo_malo = pd.concat([products, products.head(1)])

try:
    order_prods.merge(catalogo_malo, on='product_id', validate='m:1')
except pd.errors.MergeError as error:
    print('Detectado:', str(error).splitlines()[0])

Detectado: Merge keys are not unique in right dataset; not a many-to-one merge


**Interpretación:** al repetir un producto en el catálogo, `validate='m:1'` detiene el `merge` en lugar de duplicar filas en silencio.

### 4.3 Auditoría de la unión con `indicator`

In [50]:
auditoria = orders.merge(order_prods[['order_id']].drop_duplicates(),
                         on='order_id', how='left', indicator=True)
auditoria['_merge'].value_counts()

_merge
both          450046
left_only      28906
right_only         0
Name: count, dtype: int64

**Interpretación:** 450 046 pedidos tienen productos y 28 906 (6 %) no tienen ninguno registrado. Un `inner` los habría hecho desaparecer sin avisar.

### 4.4 Construcción de la tabla maestra

In [51]:
filas_inicio = len(order_prods)

maestra = order_prods.merge(products, on='product_id', how='left', validate='m:1')
print('+ products   ', maestra.shape)
maestra = maestra.merge(aisles, on='aisle_id', how='left', validate='m:1')
print('+ aisles     ', maestra.shape)
maestra = maestra.merge(departments, on='department_id', how='left', validate='m:1')
print('+ departments', maestra.shape)
maestra = maestra.merge(orders, on='order_id', how='left', validate='m:1')
print('+ orders     ', maestra.shape)

assert len(maestra) == filas_inicio

+ products    (4545007, 7)


+ aisles      (4545007, 8)


+ departments (4545007, 9)


+ orders      (4545007, 14)


**Interpretación:** la tabla maestra conserva las 4 545 007 filas de `order_prods` (la tabla de hechos) y pasa de 4 a 14 columnas. El `assert` confirma que ninguna unión creó ni perdió filas.

### 4.5 Preguntas sobre la tabla maestra

In [52]:
top = (maestra.groupby(['product_id', 'product_name']).size()
              .nlargest(5).reset_index(name='veces'))
print(top)

por_depto = (maestra.groupby('department')
                    .agg(articulos=('product_id', 'size'),
                         tasa_recompra=('reordered', 'mean'))
                    .sort_values('articulos', ascending=False))
por_depto.head(3)

   product_id            product_name  veces
0       24852                  Banana  66050
1       13176  Bag Of Organic Bananas  53297
2       21137    Organic Strawberries  37039
3       21903    Organic Baby Spinach  33971
4       47209    Organic Hass Avocado  29773


,articulos,tasa_recompra
department,,
produce,1328302,0.651140
dairy eggs,759842,0.670597
snacks,404850,0.574442


**Interpretación:** Banana es el producto más pedido (66 050 veces), seguido de Bag Of Organic Bananas (53 297). `produce` es el departamento con más artículos (1 328 302, tasa de recompra 0,651), seguido de `dairy eggs` (759 842, 0,671) y `snacks` (404 850, 0,574).

### 4.6 Apilado con `concat`

In [53]:
categorias = pd.concat([
    aisles.rename(columns={'aisle_id': 'id', 'aisle': 'nombre'}).assign(tipo='pasillo'),
    departments.rename(columns={'department_id': 'id', 'department': 'nombre'}).assign(tipo='departamento'),
], ignore_index=True)
print(categorias.shape)

(155, 3)


### 4.7 Formato ancho y largo con `pivot_table` y `melt`

In [54]:
ancho = orders.pivot_table(index='order_hour_of_day', columns='order_dow',
                           values='order_id', aggfunc='count')
largo = (ancho.reset_index()
              .melt(id_vars='order_hour_of_day', var_name='order_dow', value_name='pedidos'))
print(ancho.shape, largo.shape)

(24, 7) (168, 3)


**Interpretación:** `ancho` y `largo` contienen las mismas 168 cifras (24 horas por 7 días); el formato largo es el que prefieren `groupby` y las librerías de gráficos.

### 4.8 Guardado del resultado

In [55]:
maestra.to_parquet(SALIDA / 'instacart_maestra.parquet', index=False)
por_depto.to_csv(SALIDA / 'resumen_departamentos.csv')

**Nota:** `instacart_maestra.parquet` pesa unos 104 MB, más que el límite de 100 MB por archivo de GitHub. Por eso **no se sube al repositorio** (está en el `.gitignore`) y se regenera ejecutando este cuaderno. Los demás archivos procesados son pequeños y sí se versionan.

### Laboratorio guiado 4: tabla de análisis por producto

**Encargo:** el equipo de compras quiere una tabla con **una fila por producto**: cuántas veces se pidió, cuántos clientes lo compran, qué tan seguido se recompra y en qué posición suele ir en el carrito, para saber cuál es el producto más fiel de cada departamento.

**Paso 1: agregar la tabla maestra por producto**

In [56]:
por_producto = (maestra
    .groupby('product_id')
    .agg(veces=('order_id', 'size'),
         clientes=('user_id', 'nunique'),
         tasa_recompra=('reordered', 'mean'),
         posicion_media=('add_to_cart_order', 'mean'))
    .reset_index())
print(por_producto.shape)
por_producto.head(3)

(45573, 5)


,product_id,veces,clientes,tasa_recompra,posicion_media
0,1,280,206,0.564286,5.642857
1,2,11,11,0.000000,8.727273
2,3,42,27,0.738095,6.952381


**Interpretación:** 45 573 productos se pidieron al menos una vez (de los 49 694 del catálogo). El producto 1 se pidió 280 veces por 206 clientes, con tasa de recompra de 0,564.

**Paso 2: recuperar nombre y departamento con `merge`**

In [57]:
ficha_productos = (por_producto
    .merge(products, on='product_id', how='left', validate='1:1')
    .merge(departments, on='department_id', how='left', validate='m:1')
    [['product_id', 'product_name', 'department', 'veces', 'clientes',
      'tasa_recompra', 'posicion_media']])
print(ficha_productos.shape)
ficha_productos.loc[ficha_productos['veces'].idxmax()]

(45573, 7)


product_id           24852
product_name        Banana
department         produce
veces                66050
clientes             33274
tasa_recompra     0.844254
posicion_media    4.874815
Name: 22808, dtype: object

**Interpretación:** el producto más pedido es Banana: 66 050 veces, 33 274 clientes, tasa de recompra de 0,844 y posición media de 4,87 en el carrito.

**Paso 3: filtrar los productos con suficientes datos** (al menos 1 000 pedidos)

In [58]:
populares = ficha_productos[ficha_productos['veces'] >= 1000].copy()
print(len(populares))

739


**Interpretación:** 739 productos tienen al menos 1 000 pedidos. Con menos casos, una tasa de recompra no es confiable.

**Paso 4: clasificar dentro de cada departamento con `rank`**

In [59]:
populares['puesto'] = (populares
    .groupby('department')['tasa_recompra']
    .rank(ascending=False, method='first'))

lideres = (populares[populares['puesto'] == 1]
           .sort_values('tasa_recompra', ascending=False))
print(len(lideres))
lideres[['department', 'product_name', 'veces', 'tasa_recompra']].head(5)

18


,department,product_name,veces,tasa_recompra
41704,dairy eggs,Whole Organic Omega 3 Milk,1286,0.863142
22808,produce,Banana,66050,0.844254
29030,snacks,Extra Fancy Unsalted Mixed Nuts,1436,0.822423
42302,beverages,Zero Calorie Cola,1256,0.818471
3570,bakery,Organic Honey Sweet Whole Wheat Bread,1607,0.799004


**Interpretación:** 18 departamentos tienen al menos un producto con 1 000 pedidos, y cada uno aporta un líder. El producto más pedido no siempre es el más fiel: en lácteos, Whole Organic Omega 3 Milk tiene pocos pedidos (1 286) pero la tasa de recompra más alta (0,863).

**Paso 5: la misma respuesta con `sort_values` y `head`**

In [60]:
lideres_2 = (populares
    .sort_values('tasa_recompra', ascending=False)
    .groupby('department')
    .head(1))
print(len(lideres_2), set(lideres_2['product_id']) == set(lideres['product_id']))

18 True


**Interpretación:** ambos métodos dan los mismos 18 líderes (`True`).

**Paso 6: guardar la ficha de productos**

In [61]:
ficha_productos.to_parquet(SALIDA / 'ficha_productos.parquet', index=False)
lideres.to_csv(SALIDA / 'lideres_por_departamento.csv', index=False)

### Reto 4

Todos los ejercicios usan la tabla `maestra`.

**Ejercicio 1.** Entre los pasillos con al menos 10 000 artículos pedidos, los tres con mayor tasa de recompra, y el pasillo con la tasa más baja de todos.

In [62]:
por_pasillo = (maestra.groupby('aisle')
                      .agg(articulos=('product_id', 'size'),
                           tasa_recompra=('reordered', 'mean')))

print(por_pasillo[por_pasillo['articulos'] >= 10000].nlargest(3, 'tasa_recompra').round(4))
print()
print(por_pasillo.nsmallest(1, 'tasa_recompra').round(4))

                               articulos  tasa_recompra
aisle                                                  
milk                              125522         0.7813
water seltzer sparkling water     117133         0.7306
fresh fruits                      510101         0.7182

                   articulos  tasa_recompra
aisle                                      
spices seasonings      29454         0.1524


**Respuesta:** los tres pasillos con mayor tasa de recompra (con al menos 10 000 artículos) son `milk` (0,7813), `water seltzer sparkling water` (0,7306) y `fresh fruits` (0,7182). La tasa más baja de todos es la de `spices seasonings` (0,1524): son productos que se compran una vez y duran mucho.

**Ejercicio 2.** Participación de cada departamento en los pedidos de madrugada (0 a 5 h) frente a su participación total.

In [63]:
madrugada = maestra[maestra['order_hour_of_day'].between(0, 5)]
part_madrugada = madrugada['department'].value_counts(normalize=True).mul(100).round(2)
part_total = maestra['department'].value_counts(normalize=True).mul(100).round(2)

comparacion = pd.DataFrame({'madrugada_%': part_madrugada, 'total_%': part_total})
print(comparacion.sort_values('madrugada_%', ascending=False).head(5))
print('Top 3 madrugada:', part_madrugada.head(3).index.tolist())
print('Top 3 total:    ', part_total.head(3).index.tolist())

            madrugada_%  total_%
department                      
produce           30.21    29.23
dairy eggs        15.87    16.72
beverages          8.49     8.26
snacks             7.89     8.91
frozen             7.65     6.91
Top 3 madrugada: ['produce', 'dairy eggs', 'beverages']
Top 3 total:     ['produce', 'dairy eggs', 'snacks']


**Respuesta:** el departamento que entra en el top 3 de madrugada pero no en el total es `beverages` (8,49 % de los artículos de madrugada frente a 8,26 % del total). En el total, el tercer puesto es de `snacks` (8,91 %), que en la madrugada queda por debajo (7,89 %).

**Ejercicio 3.** Los tres productos más pedidos del departamento `snacks`.

In [64]:
(maestra[maestra['department'] == 'snacks']
    .groupby(['product_id', 'product_name']).size()
    .nlargest(3).reset_index(name='veces'))

,product_id,product_name,veces
0,13870,Lightly Salted Baked Snap Pea Crisps,3462
1,34448,Original Veggie Straws,2331
2,31040,Sea Salt Pita Chips,2275


**Respuesta:** Lightly Salted Baked Snap Pea Crisps (3 462 veces), Original Veggie Straws (2 331) y Sea Salt Pita Chips (2 275).

**Ejercicio 4.** Tabla con una fila por cliente: pedidos distintos, artículos y tasa de recompra.

In [65]:
por_cliente = (maestra.groupby('user_id')
    .agg(pedidos=('order_id', 'nunique'),
         articulos=('product_id', 'size'),
         tasa_recompra=('reordered', 'mean')))

print('Clientes:', len(por_cliente))
print('Promedio de artículos por cliente:', round(por_cliente['articulos'].mean(), 2))
print('Cliente con más artículos:', por_cliente['articulos'].idxmax(), 'con', por_cliente['articulos'].max())
por_cliente.head(3)

Clientes: 149626
Promedio de artículos por cliente: 30.38
Cliente con más artículos: 103634 con 580


,pedidos,articulos,tasa_recompra
user_id,,,
2,2,26,0.038462
4,1,2,0.000000
5,1,12,0.666667


**Respuesta:** la tabla tiene 149 626 clientes, con un promedio de 30,38 artículos por cliente. El cliente que más compró es el 103634, con 580 artículos. Son menos que los 157 437 clientes de `orders` porque la tabla maestra parte de `order_prods`: solo incluye a quienes tienen al menos un pedido con productos.

**Ejercicio 5.** Unión de `orders` con los pedidos de `order_prods` usando `how='inner'`, comparada con `how='left'`.

In [66]:
pedidos_con_productos = order_prods[['order_id']].drop_duplicates()

union_inner = orders.merge(pedidos_con_productos, on='order_id', how='inner')
union_left = orders.merge(pedidos_con_productos, on='order_id', how='left')
print('inner:', len(union_inner))
print('left: ', len(union_left))

inner: 450046
left:  478952


**Respuesta:** `inner` da 450 046 filas y `left` da 478 952. La diferencia, 28 906 pedidos, son los pedidos que no tienen ningún producto en `order_prods`: `inner` solo conserva las claves que están en ambas tablas, y `left` conserva todos los pedidos de `orders`.

**Ejercicio 6.** Tabla con los tres productos más pedidos de `produce` y los tres de `dairy eggs`, con `concat`.

In [67]:
def top3(depto):
    return (maestra[maestra['department'] == depto]
            .groupby(['product_id', 'product_name']).size()
            .nlargest(3).reset_index(name='veces')
            .assign(department=depto))

top_deptos = pd.concat([top3('produce'), top3('dairy eggs')], ignore_index=True)
top_deptos

,product_id,product_name,veces,department
0,24852,Banana,66050,produce
1,13176,Bag Of Organic Bananas,53297,produce
2,21137,Organic Strawberries,37039,produce
3,27845,Organic Whole Milk,19600,dairy eggs
4,49235,Organic Half & Half,10752,dairy eggs
5,27086,Half & Half,9588,dairy eggs


**Respuesta:** en `produce` están Banana (66 050), Bag Of Organic Bananas (53 297) y Organic Strawberries (37 039); en `dairy eggs`, Organic Whole Milk (19 600), Organic Half & Half (10 752) y Half & Half (9 588). La columna `department` indica de dónde viene cada fila.

**Ejercicio 7.** Tabla dinámica de artículos por departamento (filas) y día de la semana (columnas).

In [68]:
articulos_depto_dia = maestra.pivot_table(index='department', columns='order_dow',
                                          values='product_id', aggfunc='size')
print(articulos_depto_dia.shape)
articulos_depto_dia.loc['alcohol']

(21, 7)


order_dow
0    2397
1    3025
2    2868
3    3094
4    3656
5    3814
6    2787
Name: alcohol, dtype: int64

**Respuesta:** la tabla tiene forma (21, 7): 21 departamentos por 7 días. El día en que más se piden artículos de `alcohol` es el 5 (viernes), con 3 814, y el que menos, el 0 (domingo), con 2 397.

### Archivos procesados generados

Además de los archivos de la Lección 4, se guardan el informe de cambios de la limpieza y el perfil de hábitos de compra.

In [69]:
informe.to_csv(SALIDA / 'informe_limpieza.csv', index=False)
perfil.to_csv(SALIDA / 'perfil_habitos.csv')

for archivo in sorted(SALIDA.glob('*')):
    if archivo.name != '.gitkeep':
        print(f'{archivo.name:35s} {archivo.stat().st_size / 1e6:8.1f} MB')

ficha_productos.parquet                  1.7 MB
informe_limpieza.csv                     0.0 MB
instacart_maestra.parquet              103.6 MB
lideres_por_departamento.csv             0.0 MB
perfil_habitos.csv                       0.0 MB
resumen_departamentos.csv                0.0 MB
